# Proceso de Validacion Sucursales Clientes

Proposito del script: 
- Validar que las fechas de registro de los clientes sea posterior a las fechas de aperturas de las sucursales (se va a utilizar fecha_registro para modificar fecha_apertura).
- Modificar aquellos registros que incumplan la logica del negocio.

# Cargando los Archivos Limpios

In [1]:
import pandas as pd 
from conexiones_y_rutas import obtener_ruta_archivo
df_clientes = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_clientes.parquet"))
df_clientes_tra = df_clientes.copy()

df_sucursales = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_sucursales.parquet"))
df_sucursales_tra = df_sucursales.copy()

# Proceso de Limpieza 

Como se parte de datos que en su mayoria ya estan limpios, solo me voy a centrar en las columnas que se vean afectadas por este cambio en fecha_apertura.  

In [2]:
df_clientes_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   cliente_id                5000 non-null   int64         
 1   tipo_documento            5000 non-null   object        
 2   numero_documento          5000 non-null   object        
 3   nombres                   5000 non-null   object        
 4   apellido_paterno          5000 non-null   object        
 5   apellido_materno          5000 non-null   object        
 6   fecha_nacimiento          5000 non-null   datetime64[ns]
 7   edad                      5000 non-null   int32         
 8   genero                    5000 non-null   object        
 9   estado_civil              5000 non-null   object        
 10  nivel_educacion           5000 non-null   object        
 11  ocupacion                 5000 non-null   object        
 12  sector_economico    

In [3]:
df_sucursales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sucursal_id      24 non-null     int64         
 1   codigo_sucursal  24 non-null     object        
 2   nombre_sucursal  24 non-null     object        
 3   tipo_sucursal    24 non-null     object        
 4   ciudad           24 non-null     object        
 5   departamento     24 non-null     object        
 6   region           24 non-null     object        
 7   zona             24 non-null     object        
 8   fecha_apertura   24 non-null     datetime64[ns]
 9   estado_sucursal  24 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 2.0+ KB


In [4]:
# Selecciona las columnas a utilizar
df_clientes_veri = df_clientes_tra[['cliente_id','sucursal_id','fecha_registro']].copy()
df_sucursales_veri = df_sucursales_tra[['sucursal_id','fecha_apertura']]
# LEFT JOIN, sucursal_id 
df_merge_veri_sucursal = df_clientes_veri.merge(
    right=df_sucursales_veri,
    on="sucursal_id",
    how='left')

error_registros = df_merge_veri_sucursal[df_merge_veri_sucursal.fecha_registro < df_merge_veri_sucursal.fecha_apertura]
error_registros

,cliente_id,sucursal_id,fecha_registro,fecha_apertura
3,4,15,2010-03-12,2010-12-07
4,5,4,2013-01-08,2014-06-19
5,6,16,2015-02-09,2015-04-22
11,12,19,2010-12-20,2013-02-10
16,17,19,2012-09-14,2013-02-10
...,...,...,...,...
4961,4962,9,2010-10-19,2011-03-29
4972,4973,8,2014-12-04,2015-01-28
4976,4977,13,2012-05-07,2013-07-10
4985,4986,10,2013-09-29,2014-06-25


In [5]:
# Obtiene la fecha de registro mas pequeña para cada sucursal 
fecha_minima_registro = error_registros.groupby('sucursal_id')['fecha_registro'].min()
# La nueva fecha de apertura, es un dia anterior a la fecha minima de registro de dicho cliente
fecha_minima_registro = (fecha_minima_registro - pd.DateOffset(days=1)).to_dict()
fecha_minima_registro

{4: Timestamp('2010-02-19 00:00:00'),
 6: Timestamp('2010-02-18 00:00:00'),
 8: Timestamp('2010-02-28 00:00:00'),
 9: Timestamp('2010-06-16 00:00:00'),
 10: Timestamp('2010-03-08 00:00:00'),
 12: Timestamp('2010-05-30 00:00:00'),
 13: Timestamp('2010-02-19 00:00:00'),
 14: Timestamp('2010-03-09 00:00:00'),
 15: Timestamp('2010-03-11 00:00:00'),
 16: Timestamp('2010-02-23 00:00:00'),
 17: Timestamp('2010-05-19 00:00:00'),
 18: Timestamp('2010-02-19 00:00:00'),
 19: Timestamp('2010-02-20 00:00:00'),
 22: Timestamp('2010-03-01 00:00:00'),
 23: Timestamp('2010-02-18 00:00:00'),
 24: Timestamp('2010-02-28 00:00:00')}

In [6]:
mascara_filtro = df_sucursales_tra['sucursal_id'].isin(fecha_minima_registro)
mascara_filtro

0     False
1     False
2     False
3      True
4     False
5      True
6     False
7      True
8      True
9      True
10    False
11     True
12     True
13     True
14     True
15     True
16     True
17     True
18     True
19    False
20    False
21     True
22     True
23     True
Name: sucursal_id, dtype: bool

In [7]:
# Cambia las valores de fecha_apertura incorrectos por el nuevo recalculo
df_sucursales_tra.loc[mascara_filtro,'fecha_apertura'] = df_sucursales_tra.loc[mascara_filtro,'sucursal_id'].map(fecha_minima_registro)
df_sucursales_tra

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
0,1,SUC0001,Oficina Principal Lima,Oficina Principal,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,Activa
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,Miraflores,Lima,Lima y Callao,Urbano,2007-04-20,Activa
3,4,SUC0004,Agencia los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,Urbano,2010-02-19,Activa
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007-02-07,Activa
5,6,SUC0006,Oficina Principal Arequipa,Oficina Principal,Cayma,Arequipa,Sur,Urbano,2010-02-18,Activa
6,7,SUC0007,Agencia Arequipa 1,Punto de Atención,Arequipa,Arequipa,Sur,Urbano,2009-06-17,Activa
7,8,SUC0008,Agencia Yanahuara 2,Agencia,Yanahuara,Arequipa,Sur,Urbano,2010-02-28,Activa
8,9,SUC0009,Oficina Principal la Libertad,Oficina Principal,La Esperanza,La Libertad,Norte,Urbano,2010-06-16,Activa
9,10,SUC0010,Agencia Trujillo 1,Agencia,Trujillo,La Libertad,Norte,Urbano,2010-03-08,Activa


# Carga los Nuevos Registros y Sobrescribe el archivo original 

In [8]:
df_sucursales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sucursal_id      24 non-null     int64         
 1   codigo_sucursal  24 non-null     object        
 2   nombre_sucursal  24 non-null     object        
 3   tipo_sucursal    24 non-null     object        
 4   ciudad           24 non-null     object        
 5   departamento     24 non-null     object        
 6   region           24 non-null     object        
 7   zona             24 non-null     object        
 8   fecha_apertura   24 non-null     datetime64[ns]
 9   estado_sucursal  24 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 2.0+ KB


In [9]:
df_sucursales_tra.head()

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
0,1,SUC0001,Oficina Principal Lima,Oficina Principal,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,Activa
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,Miraflores,Lima,Lima y Callao,Urbano,2007-04-20,Activa
3,4,SUC0004,Agencia los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,Urbano,2010-02-19,Activa
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007-02-07,Activa


In [10]:
df_sucursales_tra.to_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_sucursales.parquet"),
    index=False
)